In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"
DEVICE = "cuda"

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from IPython.display import display
from tqdm import tqdm

In [ ]:
import torchvision.transforms.v2.functional as Fv2

In [ ]:
from open_vocab_mot.data import DukeMTMCItemBatch, DukeMTMCVideoDataset, collate_duke_mtmc_video_ds, DukeMTMCVideoDatasetVideoKPFBatchSampler, DukeSplit
from open_vocab_mot.definitions import DUKEMTMC_VIDEO_REID_PATH, DUKEMTMC_VIDEO_REID_SIDECAR_PATH
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from aidan_lib.models.dino_lib_compiled import DINOv3CompiledHarness

In [ ]:
duke_ds = DukeMTMCVideoDataset(
    ds_root=DUKEMTMC_VIDEO_REID_PATH,
    main_split=DukeSplit.TRAIN,
    sidecar_root=DUKEMTMC_VIDEO_REID_SIDECAR_PATH,
    load_image_pil=False,
    load_image_tensor=True,
    load_segmentations=True,
    verbose=True
)

FRAMES_PER_VIDEO = 8
duke_sampler = DukeMTMCVideoDatasetVideoKPFBatchSampler(
    duke_ds,
    batches_per_epoch=100,
    num_people_per_batch=4,
    num_views_per_person=4,
    num_frames_per_view=FRAMES_PER_VIDEO,
    allow_same_person_same_view=True,
    allow_reduced_views_per_person=False,
    allow_resampling_sample_indices=True,
    epoch_deterministic=False,
    seed = 42,
    verbose = True
)

duke_loader = DataLoader(
    dataset=duke_ds,
    collate_fn=collate_duke_mtmc_video_ds,
    batch_sampler=duke_sampler,
    num_workers=8,
    pin_memory=True
)

dino_harness = DINOv3CompiledHarness(
    checkpoint="facebook/dinov3-vits16-pretrain-lvd1689m",
    device="cuda",
    dtype=torch.bfloat16,
    max_side_len=1024,
    warmup=False
)

In [ ]:
from open_vocab_mot.data import DukeCameraId
from open_vocab_mot.data import DukePersonId
test_batch: DukeMTMCItemBatch = next(iter(duke_loader))

with torch.no_grad():
    imgs_torch = [e.to(DEVICE) for e in test_batch.frame_tensors]
    print(f"Running compiled dino")
    dino_embeddings = dino_harness.match_bool_segmentations_to_dino(imgs_torch, test_batch.segmentations)
    print(f"Finished running compiled dino")

max_viz = 16

c = 0
video_indices: dict[tuple[DukePersonId, DukeCameraId], int] = {}
video_embeddings: list[list[torch.Tensor]] = []
for sample_idx in range(len(test_batch.person_ids)):
    if len(dino_embeddings[sample_idx]) == 0:
        print(f"WARNING: sample index {sample_idx} has no dino embeddings")
        continue
    
    if c < max_viz:
        print(f"Person id: {test_batch.person_ids[sample_idx]}")
        print(f"Camera id: {test_batch.camera_ids[sample_idx]}")
        print(f"Frame shape: {test_batch.frame_tensors[sample_idx].shape}")
        print(f"Segmentation shape: {test_batch.segmentations[sample_idx].shape}")
        # Since we have different numbers of overlapping embeddings, the shape of dino embeddings changes per frame
        print(f"Dino embeddings shape {dino_embeddings[sample_idx][0].dino_embeddings.shape}")
        print()

    video_key = (int(test_batch.person_ids[sample_idx]), int(test_batch.camera_ids[sample_idx]))
    if video_key not in video_indices:
        video_index = len(video_embeddings)
        video_embeddings.append([])
        video_indices[video_key] = video_index

    video_index = video_indices[video_key]
    video_embeddings[video_index].append(
        dino_embeddings[sample_idx][0].dino_embeddings
    )

    

In [ ]:
dino_embeddings[sample_idx]

In [ ]:
print(video_indices)
print(len(video_embeddings))
print(len(video_embeddings[0]))
print(video_embeddings[0][0].size())

In [ ]:
from typing import TypedDict
from jaxtyping import Float

class ReIDFrameOutput(TypedDict):
    video_frame_contrastive_embeddings: list[Float[torch.Tensor, "frames_in_video frame_constrastive_dim"]]
    video_frame_cls_tokens: list[Float[torch.Tensor, "frames_in_video frame_transformer_dim"]]

class RIDVideoOutput(TypedDict):
    video_contrastive_embeddings: Float[torch.Tensor, "num_videos video_contrastive_dim"]
    video_cls_tokens: Float[torch.Tensor, "num_videos video_transformer_dim"]

class ReIDOutput(TypedDict):
    video_frame_contrastive_embeddings: list[Float[torch.Tensor, "frames_in_video frame_constrastive_dim"]]
    video_frame_cls_tokens: list[Float[torch.Tensor, "frames_in_video frame_transformer_dim"]]
    video_contrastive_embeddings: Float[torch.Tensor, "num_videos video_contrastive_dim"]
    video_cls_tokens: Float[torch.Tensor, "num_videos video_transformer_dim"]


class HierarchicalVideoReIDTransformer(nn.Module):
    """
    Model that processes frames individually followed by together over the whole video
    """

    def __init__(
        self,
        input_dim: int,

        frame_transformer_dim: int,
        frame_contrastive_dim: int,
        frame_num_heads: int,
        frame_num_layers: int,

        video_transformer_dim: int,
        video_contrastive_dim: int,
        video_num_heads: int,
        video_num_layers: int,

        dropout: float = 0.1
    ):
        super().__init__()

        self.input_dim = input_dim
        self.frame_transformer_dim = frame_transformer_dim
        self.frame_contrastive_dim = frame_contrastive_dim
        self.frame_num_heads = frame_num_heads
        self.frame_num_layers = frame_num_layers

        self.video_transformer_dim = video_transformer_dim
        self.video_contrastive_dim = video_contrastive_dim
        self.video_num_heads = video_num_heads
        self.video_num_layers = video_num_layers

        # The input does not need to be the same size as the frame transformer dimension so we need to project into it
        self.input_projection = nn.Linear(input_dim, frame_transformer_dim)

        # Note that position embeddings are not needed since DINO provides position embeddings
        # We also do not need temporal embeddings because we treat frames as a bag of embeddings

        # Now we can construct the frame transformer
        self.frame_cls_token = nn.Parameter(torch.randn(1, 1, frame_transformer_dim))
        frame_encoder_layer = nn.TransformerEncoderLayer(
            d_model=frame_transformer_dim,
            nhead=frame_num_heads,
            dim_feedforward=frame_transformer_dim * frame_num_heads,
            dropout=dropout,
            activation='gelu',
            batch_first=True # Crucial: Expects input as (Batch, Seq, Feature)
        )
        self.frame_transformer = nn.TransformerEncoder(frame_encoder_layer, num_layers=frame_num_layers)

        # At the output of the frame transformer we need to both project into the contrastive space
        # and into the space of the video transformer
        self.frame_contrastive_projection = nn.Linear(frame_transformer_dim, frame_contrastive_dim)
        self.frame_to_video_projection = nn.Linear(frame_transformer_dim, video_transformer_dim)

        # Then we can construct the video transformer
        self.video_cls_token = nn.Parameter(torch.randn(1, 1, video_transformer_dim))
        video_encoder_layer = nn.TransformerEncoderLayer(
            d_model=video_transformer_dim,
            nhead=video_num_heads,
            dim_feedforward=video_transformer_dim * video_num_heads,
            dropout=dropout,
            activation='gelu',
            batch_first=True # Crucial: Expects input as (Batch, Seq, Feature)
        )
        self.video_transformer = nn.TransformerEncoder(video_encoder_layer, num_layers=video_num_layers)

        # And then at the end of the video transformer we project into the contrastive space
        self.video_contrastive_projection = nn.Linear(video_transformer_dim, video_contrastive_dim)

    def embed_frames(self, video_embeddings: list[list[torch.Tensor]], device: str) -> ReIDFrameOutput:
        # In order to efficiently pass these through the transformer, we flatten 2d list into a jagged 1D list
        # and pad to be a consistent length
        # TODO: Check if there is a more efficient way to do this
        video_indices: list[int] = []
        frame_embeddings_list: list[torch.Tensor] = []
        for video_index in range(len(video_embeddings)):
            frame_embeddings = video_embeddings[video_index]
            video_indices.extend([video_index for _ in range(len(frame_embeddings))])
            frame_embeddings_list.extend(frame_embeddings)
        total_videos = len(video_embeddings)
        padded_frame_embeddings = pad_sequence(frame_embeddings_list, batch_first=True)
        total_frames = padded_frame_embeddings.size(0)
        # This is now (total_frames, max_len, input_dim)

        # We also need a mask for the attention to ignore the padding
        lengths = torch.tensor([e.size(0) for e in frame_embeddings_list], device=device)
        max_len = lengths.max()
        frame_mask = torch.arange(max_len, device=device).expand(len(padded_frame_embeddings), max_len) >= lengths.unsqueeze(1)

        # We now have what we need to run the frame level transformer
        projected_frame_embeddings = self.input_projection(padded_frame_embeddings)
        # This is now (total_frames, max_len, frame_transformer_dim)

        # Prepend [CLS] token to every sequence in the batch
        cls_tokens = self.frame_cls_token.expand(total_frames, -1, -1) # (total_frames, 1, transformer_dim)
        frame_transformer_input = torch.cat((cls_tokens, projected_frame_embeddings), dim=1) # (total_frames, max_len + 1, frame_transformer_dim)

        # The [CLS] token at index 0 is always valid, so we prepend False
        cls_mask = torch.zeros((total_frames, 1), dtype=torch.bool, device=device)
        frame_padding_mask = torch.cat((cls_mask, frame_mask), dim=1) # (total_frames, max_len + 1)

        frame_transformer_out = self.frame_transformer(frame_transformer_input, src_key_padding_mask=frame_padding_mask)

        # Extract the state of the [CLS] token
        frame_cls_out = frame_transformer_out[:, 0, :] # (total_frames, frame_transformer_dim)

        # Now we project the class tokens into the contrastive space as well
        frame_contrastive_embeddings = self.frame_contrastive_projection(frame_cls_out)
        # This is (total_frames, frame_contrastive_dim)

        # And finally we re-package back into the original videos
        video_frame_contrastive_embeddings_list: list[list[torch.Tensor]] = [[] for _ in range(total_videos)]
        video_frame_cls_tokens_list: list[list[torch.Tensor]] = [[] for _ in range(total_videos)]
        for i in range(total_frames):
            video_index = video_indices[i]
            
            video_frame_contrastive_embeddings_list[video_index].append(
                frame_contrastive_embeddings[i]
            )

            video_frame_cls_tokens_list[video_index].append(
                frame_cls_out[i]
            )

        # Stack each
        video_frame_contrastive_embeddings = [torch.stack(frame_embeddings) for frame_embeddings in video_frame_contrastive_embeddings_list]
        video_frame_cls_tokens = [torch.stack(frame_embeddings) for frame_embeddings in video_frame_cls_tokens_list]
        # These are now in the same jagged shape as the original video_embeddings
        # (num_videos, frames_in_video, embedding_size) where frames_in_video may vary between videos

        return ReIDFrameOutput(
            video_frame_contrastive_embeddings = video_frame_contrastive_embeddings,
            video_frame_cls_tokens = video_frame_cls_tokens
        )

    def embed_video_frames(self, video_frame_cls_tokens: list[torch.Tensor], device: str) -> RIDVideoOutput:
        # video_frame_cls_tokens is (num_videos, frames_in_video, frame_transformer_dim)
        # We want to pad so that the input is of size (num_videos, max_frames_in_video, frame_transformer_dim)
        padded_video_frame_tokens = pad_sequence(video_frame_cls_tokens, batch_first=True)
        num_videos = len(video_frame_cls_tokens)

        # Like with the frame level we now needs to create a mask for the transformer
        lengths = torch.tensor([len(e) for e in video_frame_cls_tokens], device=device)
        max_len = lengths.max()
        frame_mask = torch.arange(max_len, device=device).expand(len(padded_video_frame_tokens), max_len) >= lengths.unsqueeze(1)

        # Now we can project from the frame transformer dimension to the video transformer dimension
        projected_embeddings = self.frame_to_video_projection(padded_video_frame_tokens)
        # This is now (num_videos, max_frames_in_video, video_transformer_dim)

        # Prepend the [CLS] token
        cls_tokens = self.video_cls_token.expand(num_videos, -1, -1)
        video_transformer_input = torch.cat((cls_tokens, projected_embeddings), dim=1)

        # The [CLS] token at index 0 is always valid, so we prepend False
        cls_mask = torch.zeros((num_videos, 1), dtype=torch.bool, device=device)
        frame_padding_mask = torch.cat((cls_mask, frame_mask), dim=1) # (total_frames, max_len + 1)

        video_transformer_out = self.video_transformer(video_transformer_input, src_key_padding_mask=frame_padding_mask)

        # Extract the class token embeddings
        video_cls_out = video_transformer_out[:, 0, :]  # (num_videos, video_transformer_dim)

        # And now we project into the contrastive space
        video_contrastive_embeddings = self.video_contrastive_projection(video_cls_out)
        # (num_videos, video_contrastive_dim)

        return RIDVideoOutput(
            video_contrastive_embeddings = video_contrastive_embeddings,
            video_cls_tokens = video_cls_out
        )


    def forward(self, video_embeddings: list[list[torch.Tensor]]) -> ReIDOutput:
        device = video_embeddings[0][0].device

        video_frame_transformer_output = self.embed_frames(video_embeddings, device)

        video_transformer_output = self.embed_video_frames(
            video_frame_transformer_output["video_frame_cls_tokens"],
            device
        )
        
        out = ReIDOutput(
            **video_frame_transformer_output,
            **video_transformer_output
        )

        return out


In [ ]:
reid_transformer = HierarchicalVideoReIDTransformer(
    input_dim=384,
    frame_transformer_dim=256, frame_contrastive_dim=256, frame_num_heads=4, frame_num_layers=3,
    video_transformer_dim=256, video_contrastive_dim=256, video_num_heads=4, video_num_layers=3,
    dropout=0.1
).to(DEVICE)

In [ ]:
reid_output = reid_transformer(video_embeddings)

In [ ]:
list(reid_output.keys())

In [ ]:
print(len(reid_output["video_frame_contrastive_embeddings"]), [e.size() for e in reid_output["video_frame_contrastive_embeddings"]])
print(len(reid_output["video_frame_cls_tokens"]), [e.size() for e in reid_output["video_frame_cls_tokens"]])
print(reid_output["video_contrastive_embeddings"].size())

In [ ]:
class CircleLossWithUnknowns(nn.Module):
    def __init__(self, m=0.25, gamma=256):
        """
        Circle Loss with a 3-tier relationship state: Positive, Negative, and Unknown.
        
        Args:
            m (float): The margin for the circle loss (default: 0.25).
            gamma (float): The scale factor (default: 256).
        """
        super().__init__()
        self.m = m
        self.gamma = gamma
        
        # Optimums
        self.O_p = 1 + m
        self.O_n = -m
        
        # Margins
        self.Delta_p = 1 - m
        self.Delta_n = m

    def forward(self, embeddings: torch.Tensor, pos_mask: torch.Tensor, neg_mask: torch.Tensor) -> torch.Tensor:
        """
        Args:
            embeddings (torch.Tensor): Shape (N, D), where N is batch size, D is feature dim.
            pos_mask (torch.Tensor): Boolean mask of shape (N, N). True for positive pairs.
            neg_mask (torch.Tensor): Boolean mask of shape (N, N). True for negative pairs.
            
            Note: If a pair (i, j) is False in BOTH pos_mask and neg_mask, it is treated 
                  as "Unknown" and will not contribute to the loss.
        """
        # 1. Normalize embeddings to unit length for Cosine Similarity
        embeddings = F.normalize(embeddings, p=2, dim=1)

        # 2. Compute pairwise similarity matrix (N x N)
        sim_mat = torch.matmul(embeddings, embeddings.t())

        # 3. Calculate weighting factors (alpha)
        # alpha_p = ReLU(O_p - s_p) -> focuses on hard positives
        alpha_p = torch.clamp(self.O_p - sim_mat, min=0.0)
        # alpha_n = ReLU(s_n - O_n) -> focuses on hard negatives
        alpha_n = torch.clamp(sim_mat - self.O_n, min=0.0)

        # 4. Calculate weighted similarities (logits)
        logit_p = -self.gamma * alpha_p * (sim_mat - self.Delta_p)
        logit_n = self.gamma * alpha_n * (sim_mat - self.Delta_n)

        # 5. Masking
        # We replace the logits of non-relevant pairs with a very large negative number (-INF).
        # When passed into LogSumExp, exp(-INF) becomes 0, completely removing them from the loss.
        INF = 1e9
        logit_p_masked = torch.where(pos_mask, logit_p, torch.tensor(-INF, device=embeddings.device))
        logit_n_masked = torch.where(neg_mask, logit_n, torch.tensor(-INF, device=embeddings.device))

        # 6. LogSumExp aggregation per anchor
        lse_p = torch.logsumexp(logit_p_masked, dim=1)
        lse_n = torch.logsumexp(logit_n_masked, dim=1)

        # 7. Compute final loss: log(1 + exp(lse_p) * exp(lse_n)) == softplus(lse_p + lse_n)
        loss_per_anchor = F.softplus(lse_p + lse_n)

        # 8. Filter out anchors that have NO valid positive AND negative pairs
        # An anchor needs at least one positive and one negative to form a valid triad.
        valid_anchors_mask = (pos_mask.sum(dim=1) > 0) & (neg_mask.sum(dim=1) > 0)

        if valid_anchors_mask.sum() > 0:
            return loss_per_anchor[valid_anchors_mask].mean()
        else:
            # Return a zero tensor with gradients attached if the batch has no valid pairs
            return (embeddings * 0).sum()

# Test training

In [ ]:
# Instantiate the loss function and optimizer
criterion = CircleLossWithUnknowns()
optimizer = torch.optim.AdamW(reid_transformer.parameters(), lr=1e-4)

In [ ]:
reid_transformer.train()

num_epochs = 3
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, batch in enumerate(tqdm(duke_loader)):
        # 1. Extract DINO embeddings (Gradients disabled for DINO)
        with torch.no_grad():
            imgs_torch = [e.to(DEVICE) for e in batch.frame_tensors]
            dino_embeddings = dino_harness.match_bool_segmentations_to_dino(imgs_torch, batch.segmentations)
            
        # 2. Group embeddings by video (person_id, camera_id) and track person_ids
        video_indices: dict[tuple[int, int], int] = {}
        video_embeddings: list[list[torch.Tensor]] = []
        video_person_ids: list[int] = []
        
        for sample_idx in range(len(batch.person_ids)):
            # Safeguard: skip frame if no segmentations/embeddings were found
            if len(dino_embeddings[sample_idx]) == 0:
                continue
                
            video_key = (int(batch.person_ids[sample_idx]), int(batch.camera_ids[sample_idx]))
            
            if video_key not in video_indices:
                video_index = len(video_embeddings)
                video_embeddings.append([])
                video_indices[video_key] = video_index
                video_person_ids.append(int(batch.person_ids[sample_idx]))
                
            video_index = video_indices[video_key]
            # Assuming index 0 is the correct person embedding for this test
            video_embeddings[video_index].append(
                dino_embeddings[sample_idx][0].dino_embeddings
            )
            
        # Skip batch if we don't have enough valid videos to form pairs
        if len(video_embeddings) < 2:
            continue
            
        # 3. Create Positive and Negative Masks (N x N)
        N = len(video_person_ids)
        pos_mask = torch.zeros((N, N), dtype=torch.bool, device=DEVICE)
        neg_mask = torch.zeros((N, N), dtype=torch.bool, device=DEVICE)
        
        for i in range(N):
            for j in range(N):
                if i == j:
                    continue # Ignore self pairings
                if video_person_ids[i] == video_person_ids[j]:
                    pos_mask[i, j] = True
                else:
                    neg_mask[i, j] = True
                    
        # 4. Forward Pass through REID Transformer
        optimizer.zero_grad()
        reid_output = reid_transformer(video_embeddings)
        
        # We exclusively optimize the video contrastive embeddings
        embeddings = reid_output["video_contrastive_embeddings"]
        
        # 5. Compute Loss
        loss = criterion(embeddings, pos_mask, neg_mask)
        
        # 6. Backward Pass and Optimize
        loss.backward()
        optimizer.step()
        
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}: Loss = {loss.item():.4f}")


# Multi-loss training

In [ ]:
import matplotlib.pyplot as plt

# Instantiate the loss function and optimizer
criterion = CircleLossWithUnknowns()
optimizer = torch.optim.AdamW(reid_transformer.parameters(), lr=1e-4)

reid_transformer.train()

# Hyperparameter to tune the weight between video loss and frame loss
# total_loss = (1 - frame_loss_weight) * video_loss + frame_loss_weight * frame_loss
frame_loss_weight = 0.25

# History trackers
history = {
    "video_loss": [],
    "frame_loss": [],
    "total_loss": []
}

num_epochs = 3
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, batch in enumerate(tqdm(duke_loader)):
        # 1. Extract DINO embeddings (Gradients disabled for DINO)
        with torch.no_grad():
            imgs_torch = [e.to(DEVICE) for e in batch.frame_tensors]
            dino_embeddings = dino_harness.match_bool_segmentations_to_dino(imgs_torch, batch.segmentations)
            
        # 2. Group embeddings by video (person_id, camera_id) and track person_ids
        video_indices: dict[tuple[int, int], int] = {}
        video_embeddings: list[list[torch.Tensor]] = []
        video_person_ids: list[int] = []
        
        for sample_idx in range(len(batch.person_ids)):
            # Safeguard: skip frame if no segmentations/embeddings were found
            if len(dino_embeddings[sample_idx]) == 0:
                continue
                
            video_key = (int(batch.person_ids[sample_idx]), int(batch.camera_ids[sample_idx]))
            
            if video_key not in video_indices:
                video_index = len(video_embeddings)
                video_embeddings.append([])
                video_indices[video_key] = video_index
                video_person_ids.append(int(batch.person_ids[sample_idx]))
                
            video_index = video_indices[video_key]
            # Assuming index 0 is the correct person embedding for this test
            video_embeddings[video_index].append(
                dino_embeddings[sample_idx][0].dino_embeddings
            )
            
        # Skip batch if we don't have enough valid videos to form pairs
        if len(video_embeddings) < 2:
            continue
            
        # 3. Create Positive and Negative Masks
        # --- Video level ---
        video_pids = torch.tensor(video_person_ids, device=DEVICE)
        video_pos_mask = (video_pids.unsqueeze(0) == video_pids.unsqueeze(1))
        video_neg_mask = ~video_pos_mask
        video_pos_mask.fill_diagonal_(False)
        video_neg_mask.fill_diagonal_(False)
        
        # --- Frame level ---
        # Flatten the person IDs so they align with the concatenated frame embeddings
        flat_frame_person_ids = []
        for v_idx, frames in enumerate(video_embeddings):
            flat_frame_person_ids.extend([video_person_ids[v_idx]] * len(frames))
            
        frame_pids = torch.tensor(flat_frame_person_ids, device=DEVICE)
        frame_pos_mask = (frame_pids.unsqueeze(0) == frame_pids.unsqueeze(1))
        frame_neg_mask = ~frame_pos_mask
        frame_pos_mask.fill_diagonal_(False)
        frame_neg_mask.fill_diagonal_(False)
        
        # 4. Forward Pass through REID Transformer
        optimizer.zero_grad()
        reid_output = reid_transformer(video_embeddings)
        
        # 5. Compute Losses
        # Video contrastive loss (same as before)
        video_embeddings_out = reid_output["video_contrastive_embeddings"]
        video_loss = criterion(video_embeddings_out, video_pos_mask, video_neg_mask)
        
        # Frame contrastive loss (new)
        # We concatenate all frame contrastive embeddings into a single (total_frames, dim) tensor
        frame_embeddings_out = torch.cat(reid_output["video_frame_contrastive_embeddings"], dim=0)
        frame_loss = criterion(frame_embeddings_out, frame_pos_mask, frame_neg_mask)
        
        # Combine losses using the hyperparameter
        total_loss = (1.0 - frame_loss_weight) * video_loss + frame_loss_weight * frame_loss
        
        # 6. Backward Pass and Optimize
        total_loss.backward()
        optimizer.step()
        
        # Track history
        history["video_loss"].append(video_loss.item())
        history["frame_loss"].append(frame_loss.item())
        history["total_loss"].append(total_loss.item())
        
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}: Total Loss = {total_loss.item():.4f} "
                  f"(Video: {video_loss.item():.4f}, Frame: {frame_loss.item():.4f})")

# Plotting the history
plt.figure(figsize=(10, 6))
plt.plot(history["video_loss"], label="Video Loss", alpha=0.7)
plt.plot(history["frame_loss"], label="Frame Loss", alpha=0.7)
plt.plot(history["total_loss"], label="Total Loss", linewidth=2)
plt.xlabel("Batch Index")
plt.ylabel("Loss")
plt.title(f"Training Loss over Time (Frame Weight = {frame_loss_weight})")
plt.legend()
plt.grid(True)
plt.show()


# Evaluation

There are two sets used during evaluation:
*Query*: At path `/query`. Same structure as train, but only has one view per individual. Used to compare to the gallery to get accuracy.
> Question: Are the individuals unique from those seen in the train set?

*Gallery*: At path `/gallery`. Same structure as train. Used as a bank of possible identities to compare against.

Evaluation goes as follows:
Embed all frames from each video in both query and gallery.
Randomly sample frames from each video of the same length as was used to train the transformer and create N video embeddings for each video.

Create data structures that contain the frame embeddings and the video embeddings that allow for fast top k lookup using cosine similarity.

But actually, since we have relatively few videos, we can just compute the similarity to every video embedding in the gallery and do something like take the average similarity between the video embeddings.

In [ ]:
gallery_ds = DukeMTMCVideoDataset(
    ds_root=DUKEMTMC_VIDEO_REID_PATH,
    main_split=DukeSplit.GALLERY,
    sidecar_root=DUKEMTMC_VIDEO_REID_SIDECAR_PATH,
    load_image_pil=False,
    load_image_tensor=True,
    load_segmentations=True,
    verbose=True
)

query_ds = DukeMTMCVideoDataset(
    ds_root=DUKEMTMC_VIDEO_REID_PATH,
    main_split=DukeSplit.QUERY,
    sidecar_root=DUKEMTMC_VIDEO_REID_SIDECAR_PATH,
    load_image_pil=False,
    load_image_tensor=True,
    load_segmentations=True,
    verbose=True
)

In [ ]:
import random

@torch.no_grad()
def process_duke_ds(ds: DukeMTMCVideoDataset, model: HierarchicalVideoReIDTransformer, batch_size: int = 64, video_batch_size=64, num_video_embeddings_per_video: int = 1):
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_duke_mtmc_video_ds,
        pin_memory=True,
        num_workers=8
    )

    num_frames = len(ds)
    num_videos = sum(len(ds.frame_map[person_id]) for person_id in ds.frame_map)
    frame_embedding_size = model.frame_transformer_dim
    frame_contrastive_embedding_size = model.frame_contrastive_dim
    video_contrastive_embedding_size = model.video_contrastive_dim

    print("Creating buffers")
    frame_data = []
    frame_embeddings = torch.zeros((num_frames, frame_embedding_size), device="cpu", dtype=torch.float32)
    frame_contrastive_embeddings = torch.zeros((num_frames, frame_contrastive_embedding_size), device="cpu", dtype=torch.float32)
    # For frame_embeddings and frame_contrastive_embeddings we may underfill due to skipped frames. Length must come from frame_data.
    video_contrastive_embeddings = torch.zeros((num_videos * num_video_embeddings_per_video, video_contrastive_embedding_size), device="cpu", dtype=torch.float32)
    print("Created buffers")

    progress = tqdm(loader)
    frame_count = 0
    batch: DukeMTMCItemBatch
    for batch in progress:
        # We make a fake "video" made of all these frames since we are just using the frame embedder at this stage
        progress.set_description_str("Running DINO")
        imgs_torch = [e.to(DEVICE) for e in batch.frame_tensors]
        dino_embeddings = dino_harness.match_bool_segmentations_to_dino(imgs_torch, batch.segmentations)
        
        progress.set_description_str("Extracting DINO embeddings")
        person_ids, camera_ids = batch.person_ids, batch.camera_ids
        batch_frame_data = []
        dino_embedding_video: list[torch.Tensor] = []
        for person_id, camera_id, frame_dino_embedding in zip(person_ids, camera_ids, dino_embeddings):
            person_id = int(person_id)
            camera_id = int(camera_id)

            # print(person_id, camera_id)

            if len(frame_dino_embedding) == 0:
                print(f"Warning: No segmentation found for a frame in person {person_id} camera {camera_id}")
                continue

            batch_frame_data.append((person_id, camera_id))
            dino_embedding_video.append(frame_dino_embedding[0].dino_embeddings)

        # Now we can extract the frame embeddings
        progress.set_description_str("Embedding frames")
        batch_frame_embedding_data = model.embed_frames([dino_embedding_video], device=DEVICE)

        batch_frame_contrastive_embeddings = batch_frame_embedding_data["video_frame_contrastive_embeddings"][0].to(device="cpu")
        batch_frame_cls_tokens = batch_frame_embedding_data["video_frame_cls_tokens"][0].to(device="cpu")

        assert len(batch_frame_contrastive_embeddings) == len(batch_frame_cls_tokens) == len(batch_frame_data)
        for frame_contrastive_embedding, frame_cls_token, this_frame_data in zip(batch_frame_contrastive_embeddings, batch_frame_cls_tokens, batch_frame_data):
            # print(this_frame_data)
            frame_embeddings[frame_count] = frame_cls_token
            frame_contrastive_embeddings[frame_count] = frame_contrastive_embedding
            frame_data.append(this_frame_data)

            frame_count += 1

    # We now want to process these frame embeddings into video embeddings
    # To do this we first need a map from person to camera to list of frame indices so that we can build the batches
    frame_index_map: dict[DukePersonId, dict[DukeCameraId, list[int]]] = {}
    for frame_idx, (person_id, camera_id) in enumerate(frame_data):
        # print(person_id, camera_id)
        if person_id not in frame_index_map:
            frame_index_map[person_id] = {}
        camera_index_map = frame_index_map[person_id]

        if camera_id not in camera_index_map:
            camera_index_map[camera_id] = []

        frame_list = camera_index_map[camera_id]
        frame_list.append(frame_idx)

    # Now we convert into the format that we can pass into the video embedding layer
    video_embedding_index_map: dict[DukePersonId, dict[DukeCameraId, list[int]]] = {}

    rng = random.Random()
    video_count = 0
    progress = tqdm(total=num_videos * num_video_embeddings_per_video)
    batch_video_data_buffer: list = []
    batch_frame_buffer: list[torch.Tensor] = []
    for person_id, camera_index_map in frame_index_map.items():
        for camera_id, frame_indices in camera_index_map.items():
            num_frames = min(len(frame_indices), FRAMES_PER_VIDEO)
            for _ in range(num_video_embeddings_per_video):
                # Sample a random set of frames of size FRAMES_PER_VIDEO without replacement to serve as the video
                sampled_frame_indices = rng.sample(frame_indices, k=num_frames)

                video_frame_tensor = torch.stack([frame_embeddings[i] for i in sampled_frame_indices]).to(device=DEVICE)

                batch_video_data_buffer.append((person_id, camera_id, video_count))
                batch_frame_buffer.append(video_frame_tensor)

                if person_id not in video_embedding_index_map:
                    video_embedding_index_map[person_id] = {}
                
                if camera_id not in video_embedding_index_map[person_id]:
                    video_embedding_index_map[person_id][camera_id] = []

                if len(batch_video_data_buffer) == video_batch_size:
                    video_embedding_data = model.embed_video_frames(batch_frame_buffer, device=DEVICE)
                    batch_video_contrastive_embeddings = video_embedding_data["video_contrastive_embeddings"]

                    for (person_id, camera_id, video_idx), video_contrastive_embedding in zip(batch_video_data_buffer, batch_video_contrastive_embeddings):
                        video_embedding_index_map[person_id][camera_id].append(video_idx)
                        video_contrastive_embeddings[video_idx] = video_contrastive_embedding.cpu()

                    batch_video_data_buffer.clear()
                    batch_frame_buffer.clear()

                video_count += 1
                progress.update(1)
    
    if len(batch_video_data_buffer) > 0:
        video_embedding_data = model.embed_video_frames(batch_frame_buffer, device=DEVICE)
        batch_video_contrastive_embeddings = video_embedding_data["video_contrastive_embeddings"]

        for (person_id, camera_id, video_idx), video_contrastive_embedding in zip(batch_video_data_buffer, batch_video_contrastive_embeddings):
            video_embedding_index_map[person_id][camera_id].append(video_idx)
            video_contrastive_embeddings[video_idx] = video_contrastive_embedding.cpu()

        batch_video_data_buffer.clear()
        batch_frame_buffer.clear()

    # Slice down tensors to their true size
    true_num_frames = len(frame_data)
    true_num_videos = video_count

    frame_embeddings = frame_embeddings[:true_num_frames]
    frame_contrastive_embeddings = frame_contrastive_embeddings[:true_num_frames]
    video_contrastive_embeddings = video_contrastive_embeddings[:true_num_videos]
    
    return {
        "frame_index_map": frame_index_map,
        "video_embedding_index_map": video_embedding_index_map,
        "frame_data": frame_data,
        "frame_embeddings": frame_embeddings,
        "frame_contrastive_embeddings": frame_contrastive_embeddings,
        "video_contrastive_embeddings": video_contrastive_embeddings
    }


In [ ]:
processed_query_ds = process_duke_ds(query_ds, reid_transformer, batch_size=512)

In [ ]:
processed_query_ds["video_contrastive_embeddings"].shape

In [ ]:
import torch
import torch.nn.functional as F

def group_embeddings(processed_ds, num_embeddings_per_video):
    """
    Stacks and aligns embeddings to shape (num_videos, num_embeddings_per_video, D)
    based on the index map, maintaining the correct order.
    """
    video_embeddings = processed_ds["video_contrastive_embeddings"]
    video_map = processed_ds["video_embedding_index_map"]
    
    grouped_list = []
    metadata = [] # List of (person_id, camera_id)
    
    for person_id, camera_map in video_map.items():
        for camera_id, indices in camera_map.items():
            assert len(indices) == num_embeddings_per_video, \
                f"Expected {num_embeddings_per_video} embeddings, but got {len(indices)}"
            video_embs = torch.stack([video_embeddings[idx] for idx in indices])
            grouped_list.append(video_embs)
            metadata.append((person_id, camera_id))
            
    return torch.stack(grouped_list), metadata

@torch.no_grad()
def evaluate_duke_reid(
    processed_query: dict,
    processed_gallery: dict,
    num_embeddings_per_video: int = 1,
    sim_aggregation: str = "max", # "max" or "mean"
    device: str = "cuda"
):
    """
    Computes similarities and top-k retrieval accuracies (CMC / mAP) 
    over the Query and Gallery sets.
    """
    # 1. Align and group embeddings
    query_embs, query_meta = group_embeddings(processed_query, num_embeddings_per_video)
    gallery_embs, gallery_meta = group_embeddings(processed_gallery, num_embeddings_per_video)
    
    # 2. Move to device & Normalize for Cosine Similarity
    query_embs = F.normalize(query_embs.to(device), p=2, dim=-1)
    gallery_embs = F.normalize(gallery_embs.to(device), p=2, dim=-1)
    
    # 3. Compute pairwise similarities between all query and gallery embeddings
    # query_embs: (N_Q, M, D), gallery_embs: (N_G, M, D) -> (N_Q, N_G, M, M)
    print("Computing embedding-level pairwise similarities...")
    pairwise_sims = torch.einsum('qmd,gnd->qgmn', query_embs, gallery_embs)
    
    # 4. Aggregate across the M embeddings of each video pair
    print(f"Aggregating video-to-video similarities using '{sim_aggregation}'...")
    if sim_aggregation == "max":
        video_sims = pairwise_sims.max(dim=-1)[0].max(dim=-1)[0] # Shape: (N_Q, N_G)
    elif sim_aggregation == "mean":
        video_sims = pairwise_sims.mean(dim=(-2, -1)) # Shape: (N_Q, N_G)
    else:
        raise ValueError(f"Unknown sim_aggregation: {sim_aggregation}")
    
    # Move similarity matrix back to CPU
    video_sims = video_sims.cpu()
    
    # Extract metadata arrays
    q_pids = torch.tensor([meta[0] for meta in query_meta])
    q_cids = torch.tensor([meta[1] for meta in query_meta])
    g_pids = torch.tensor([meta[0] for meta in gallery_meta])
    g_cids = torch.tensor([meta[1] for meta in gallery_meta])
    
    N_Q = len(query_meta)
    
    # ==========================================
    # Metric 1: Standard Video-to-Video ReID Evaluation
    # ==========================================
    print("\n--- Standard Video-to-Video Evaluation (Standard CMC/mAP) ---")
    cmc_video = torch.zeros(N_Q)
    ap_video = torch.zeros(N_Q)
    valid_queries_video = 0
    
    for q_idx in range(N_Q):
        pid = q_pids[q_idx]
        cid = q_cids[q_idx]
        
        # Filter out same camera and same identity matches (trivial matches)
        exclude_mask = (g_pids == pid) & (g_cids == cid)
        
        sims = video_sims[q_idx].clone()
        sims[exclude_mask] = -1e9 # Set excluded elements to -inf
        
        truth_indices = (g_pids == pid) & ~exclude_mask
        num_g_truth = truth_indices.sum().item()
        
        if num_g_truth == 0:
            continue
            
        valid_queries_video += 1
        
        # Rank gallery items
        sorted_indices = torch.argsort(sims, descending=True)
        sorted_truth = truth_indices[sorted_indices]
        
        # CMC
        first_match_rank = torch.where(sorted_truth)[0][0].item()
        cmc_video[first_match_rank:] += 1
        
        # AP
        correct_ranks = torch.where(sorted_truth)[0]
        precision_at_ranks = (torch.arange(1, len(correct_ranks) + 1, dtype=torch.float32) / 
                               (correct_ranks.float() + 1))
        ap_video[q_idx] = precision_at_ranks.mean()
        
    cmc_video = cmc_video / valid_queries_video
    map_video = ap_video.sum() / valid_queries_video
    
    print(f"Rank-1 Accuracy:  {cmc_video[0].item() * 100:.2f}%")
    print(f"Rank-5 Accuracy:  {cmc_video[4].item() * 100:.2f}%")
    print(f"Rank-10 Accuracy: {cmc_video[9].item() * 100:.2f}%")
    print(f"mAP:              {map_video.item() * 100:.2f}%")
    
    # ==========================================
    # Metric 2: Person-to-Identity Evaluation
    # ==========================================
    print("\n--- Person-to-Identity Evaluation (Max over gallery videos) ---")
    gallery_people = sorted(list(set(g_pids.tolist())))
    cmc_identity = torch.zeros(N_Q)
    valid_queries_identity = 0
    
    for q_idx in range(N_Q):
        pid = q_pids[q_idx].item()
        cid = q_cids[q_idx].item()
        
        id_sims = []
        id_list = []
        
        for g_pid in gallery_people:
            # Find all videos of this person
            person_mask = (g_pids == g_pid)
            # Exclude same camera match if it's the target query subject
            if g_pid == pid:
                person_mask = person_mask & (g_cids != cid)
                
            if person_mask.sum() == 0:
                continue
                
            # Aggregate via Max similarity to this identity's videos
            person_sim = video_sims[q_idx, person_mask].max().item()
            id_sims.append(person_sim)
            id_list.append(g_pid)
            
        if pid not in id_list:
            continue
            
        valid_queries_identity += 1
        
        id_sims = torch.tensor(id_sims)
        id_list = torch.tensor(id_list)
        
        # Rank identities
        sorted_indices = torch.argsort(id_sims, descending=True)
        sorted_ids = id_list[sorted_indices]
        
        first_match_rank = torch.where(sorted_ids == pid)[0][0].item()
        cmc_identity[first_match_rank:] += 1
        
    cmc_identity = cmc_identity / valid_queries_identity
    
    print(f"Rank-1 Accuracy:  {cmc_identity[0].item() * 100:.2f}%")
    print(f"Rank-5 Accuracy:  {cmc_identity[4].item() * 100:.2f}%")
    print(f"Rank-10 Accuracy: {cmc_identity[9].item() * 100:.2f}%")
    
    return {
        "video": {
            "cmc": cmc_video,
            "mAP": map_video,
            "rank_1": cmc_video[0].item(),
            "rank_5": cmc_video[4].item(),
            "rank_10": cmc_video[9].item()
        },
        "identity": {
            "cmc": cmc_identity,
            "rank_1": cmc_identity[0].item(),
            "rank_5": cmc_identity[4].item(),
            "rank_10": cmc_identity[9].item()
        }
    }


In [ ]:
# Number of random frame samples per video (change to e.g. 3 or 5 for multiple embeddings)
num_embeddings = 3

print("Processing Query Dataset...")
processed_query_ds = process_duke_ds(
    query_ds, 
    reid_transformer, 
    batch_size=512, 
    num_video_embeddings_per_video=num_embeddings
)

print("\nProcessing Gallery Dataset...")
processed_gallery_ds = process_duke_ds(
    gallery_ds, 
    reid_transformer, 
    batch_size=512, 
    num_video_embeddings_per_video=num_embeddings
)

# Run the evaluation metrics
results = evaluate_duke_reid(
    processed_query_ds, 
    processed_gallery_ds, 
    num_embeddings_per_video=num_embeddings,
    sim_aggregation="max",
    device=DEVICE
)
